## 🥇 Gold Layer - Analytics & Crime Pattern Insights
This notebook reads from the Silver Delta table and runs Spark SQL analytics to uncover real crime patterns across Chicago. These aggregated insights form our Gold layer — business-ready data for dashboards and ML models.

In [0]:
# Read from Silver Delta table
df_silver = spark.read.format("delta").table("workspace.default.silver_crimes")

print(f"Records loaded from Silver: {df_silver.count()}")

Records loaded from Silver: 1269490


### 🔍 Analysis 1 - Top Crime Types in Chicago
Which crimes happen most frequently across the city?

In [0]:
df_top_crimes = spark.sql("""
    SELECT 
        Primary_Type,
        COUNT(*) as total_crimes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage,
        SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) as total_arrests,
        ROUND(SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as arrest_rate
    FROM workspace.default.silver_crimes
    GROUP BY Primary_Type
    ORDER BY total_crimes DESC
    LIMIT 10
""")

display(df_top_crimes)

Primary_Type,total_crimes,percentage,total_arrests,arrest_rate
THEFT,282088,22.22,17061,6.05
BATTERY,227642,17.93,37360,16.41
CRIMINAL DAMAGE,144787,11.41,5322,3.68
ASSAULT,115314,9.08,12262,10.63
MOTOR VEHICLE THEFT,105927,8.34,3178,3.00
DECEPTIVE PRACTICE,83141,6.55,2334,2.81
OTHER OFFENSE,82725,6.52,13597,16.44
ROBBERY,44236,3.48,3165,7.15
BURGLARY,44118,3.48,2195,4.98
WEAPONS VIOLATION,41079,3.24,26138,63.63


### 📅 Analysis 2 - Crime by Hour of Day
When do crimes peak during the day? Identifying dangerous hours.

In [0]:
df_by_hour = spark.sql("""
    SELECT 
        Hour,
        COUNT(*) as total_crimes,
        ROUND(SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as arrest_rate,
        SUM(CASE WHEN Domestic = true THEN 1 ELSE 0 END) as domestic_crimes
    FROM workspace.default.silver_crimes
    GROUP BY Hour
    ORDER BY Hour ASC
""")

display(df_by_hour)

Hour,total_crimes,arrest_rate,domestic_crimes
0,91041,10.11,18790
1,42038,13.94,10485
2,37403,12.28,9127
3,31985,10.52,7429
4,26181,9.34,5802
5,22718,8.31,4899
6,24319,9.83,5028
7,31982,10.75,6882
8,43269,10.94,9174
9,53371,10.02,10666


### 📍 Analysis 3 - Most Dangerous Districts
Which Chicago districts have the highest crime rates and lowest arrest rates?

In [0]:
df_by_district = spark.sql("""
    SELECT 
        District,
        COUNT(*) as total_crimes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_of_all_crimes,
        ROUND(SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as arrest_rate,
        SUM(CASE WHEN Domestic = true THEN 1 ELSE 0 END) as domestic_crimes,
        ROUND(AVG(Latitude), 6) as avg_latitude,
        ROUND(AVG(Longitude), 6) as avg_longitude
    FROM workspace.default.silver_crimes
    GROUP BY District
    ORDER BY total_crimes DESC
""")

display(df_by_district)

District,total_crimes,pct_of_all_crimes,arrest_rate,domestic_crimes,avg_latitude,avg_longitude
8,81284,6.40,11.72,16980,41.778968,-87.717858
12,76522,6.03,8.54,8745,41.880129,-87.668152
6,76183,6.00,12.69,20804,41.745582,-87.631045
4,71794,5.66,10.17,18917,41.732306,-87.563282
11,70683,5.57,25.54,16296,41.881837,-87.718827
1,68495,5.40,16.48,4872,41.872129,-87.628389
18,65863,5.19,13.31,4387,41.900618,-87.633183
19,65639,5.17,8.75,4942,41.94741,-87.658335
3,65373,5.15,11.61,17773,41.770965,-87.595386
25,65033,5.12,14.61,13836,41.91985,-87.753758


### 📅 Analysis 4 - Crime Trends by Season & Weekend
Do crimes spike in summer? Are weekends more dangerous than weekdays?

In [0]:
df_season = spark.sql("""
    SELECT 
        Season,
        Is_Weekend,
        COUNT(*) as total_crimes,
        ROUND(SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as arrest_rate,
        ROUND(SUM(CASE WHEN Is_Night = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as night_crime_pct
    FROM workspace.default.silver_crimes
    GROUP BY Season, Is_Weekend
    ORDER BY total_crimes DESC
""")

display(df_season)

Season,Is_Weekend,total_crimes,arrest_rate,night_crime_pct
Spring,false,237513,13.89,36.46
Summer,false,229989,12.68,40.13
Fall,false,221160,12.34,37.51
Winter,false,214007,14.20,35.56
Spring,true,95726,14.44,44.60
Summer,true,95479,12.77,47.40
Fall,true,90133,12.55,45.81
Winter,true,85483,14.70,43.50


### 💾 Save Gold Aggregates to Delta Tables
Save all 4 analyses as Gold Delta tables for dashboard and ML use.

In [0]:
# Save all 4 gold aggregates as Delta tables
df_top_crimes.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_crimes")

df_by_hour.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_crimes_by_hour")

df_by_district.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_crimes_by_district")

df_season.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_crimes_by_season")

print("✅ All Gold tables saved successfully!")
print("   - workspace.default.gold_top_crimes")
print("   - workspace.default.gold_crimes_by_hour")
print("   - workspace.default.gold_crimes_by_district")
print("   - workspace.default.gold_crimes_by_season")

✅ All Gold tables saved successfully!
   - workspace.default.gold_top_crimes
   - workspace.default.gold_crimes_by_hour
   - workspace.default.gold_crimes_by_district
   - workspace.default.gold_crimes_by_season
